# Platform walkthrough

A guided tour of what each layer produced. Run `make pipeline` first, then open this notebook with:

```bash
uv run --with jupyter jupyter lab notebooks/
```

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make `src` importable from notebooks/

import pandas as pd

from src.utils import db
from src.utils.paths import Paths

paths = Paths()

## 1. Data Engineering: what did validation catch?

In [ ]:
report = json.loads(paths.validation_report_json.read_text())
pd.DataFrame(report["tables"]).T

In [ ]:
pd.read_csv(paths.rejected_records).groupby(["table", "rule", "action"]).size().rename(
    "records"
).reset_index()

## 2. Features: one row per application, using only past information

In [ ]:
features = pd.read_parquet(paths.features)
features.groupby("split")["default_flag"].agg(["size", "mean"])

In [ ]:
# Do the features carry signal? Default rate by repayment history bucket
labelled = features[features["split"].isin(["train", "validation", "test"])]
labelled.groupby(pd.cut(labelled["late_payment_count_90d"], [-1, 0, 1, 10]), observed=True)[
    "default_flag"
].agg(["size", "mean"])

## 3. AI Engineering: model performance and thresholds

In [ ]:
metrics = json.loads(paths.training_metrics.read_text())
pd.DataFrame(metrics["model_comparison"])

In [ ]:
pd.DataFrame({"model": metrics["test"], "manual underwriter": metrics["manual_underwriter_test"]}).loc[
    ["auto_approval_rate", "bad_rate_among_approved", "recall", "precision", "roc_auc"]
]

## 4. Backend Engineering: score an application with the same code the API uses

In [ ]:
from src.models.predict import CreditScorer

scorer = CreditScorer.load()
example = pd.DataFrame(
    [
        {
            "requested_amount": 25000,
            "recent_procurement_value": 180000,
            "late_payment_count_90d": 1,
            "repayment_rate_180d": 0.91,
            "stock_turnover_ratio": 1.3,
        }
    ]
)
scorer.score_frame(example)[
    ["score", "decision", "recommended_limit", "recommended_tenor_months", "reason_codes"]
]

## 5. Monitoring: what changed after launch, and who owns it?

In [ ]:
checks = pd.read_csv(paths.monitoring_checks)
checks.pivot_table(index="owner_layer", columns="status", values="check_name", aggfunc="count", fill_value=0)

In [ ]:
checks[checks["status"] == "alert"][["owner_layer", "check_name", "subject", "explanation"]]

In [ ]:
db.read_sql(
    "SELECT decision, COUNT(*) AS n, ROUND(AVG(score), 3) AS avg_score FROM model_scores GROUP BY decision"
)